# Consolidated Rejected Shrimp Shipments (2020-2025)

This notebook cell builds a consolidated yearly table of rejected shrimp shipments by:
- Market: FDA, EU
- Country: Vietnam (VI), Ecuador (EC), India (IN)
- Barrier: SPS, TBT

It outputs:
1. A Markdown table for documentation
2. A CSV block for Excel integration

Note: If 2025 is partial, the output adds a `2025*` notation with the latest available date.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# -----------------------------
# 1) Locate input files robustly
# -----------------------------
def resolve_base_dir() -> Path:
    """Find folder that contains FDA/ and EU/ directories."""
    candidates = [
        Path.cwd(),
        Path.cwd() / "KTPT",
        Path.cwd().parent,
    ]
    for c in candidates:
        if (c / "FDA").exists() and (c / "EU").exists():
            return c
    raise FileNotFoundError("Could not locate folder containing FDA/ and EU/ datasets.")

base_dir = resolve_base_dir()
fda_file = base_dir / "FDA" / "FDA_ShrimpPrawns_2020_2025_filtered.csv"
eu_file = base_dir / "EU" / "EU_RASFF_ShrimpPrawns_2020_2025_filtered.csv"

# -----------------------------
# 2) Read and normalize datasets
# -----------------------------
fda = pd.read_csv(fda_file)
eu = pd.read_csv(eu_file)

# FDA normalization
fda = fda.rename(columns={
    "country_name": "country",
    "REFUSAL_DATE_PARSED": "date",
    "sps_tbt_category": "barrier",
})
fda["date"] = pd.to_datetime(fda["date"], errors="coerce")

# Keep target countries only
valid_countries = ["Vietnam", "Ecuador", "India"]
fda = fda[fda["country"].isin(valid_countries)].copy()

# Convert SPS+TBT into two counts (same rejected shipment contributes to both categories)
fda["barrier"] = fda["barrier"].replace({"SPS+TBT": "SPS|TBT"})
fda = fda[fda["barrier"].isin(["SPS", "TBT", "SPS|TBT"])].copy()
fda["barrier"] = fda["barrier"].str.split("|", regex=False)
fda = fda.explode("barrier")

fda["year"] = fda["date"].dt.year
fda = fda[(fda["year"] >= 2020) & (fda["year"] <= 2025)].copy()
fda["market"] = "FDA"

# EU normalization
eu = eu.rename(columns={
    "_date": "date",
    "category_class": "barrier",
})
eu["date"] = pd.to_datetime(eu["date"], errors="coerce")
eu = eu[eu["country"].isin(valid_countries)].copy()
eu = eu[eu["barrier"].isin(["SPS", "TBT"])].copy()
eu["year"] = eu["date"].dt.year
eu = eu[(eu["year"] >= 2020) & (eu["year"] <= 2025)].copy()
eu["market"] = "EU"

# -----------------------------
# 3) Build consolidated table
# -----------------------------
country_code = {
    "Vietnam": "VI",
    "Ecuador": "EC",
    "India": "IN",
}

combined = pd.concat([
    fda[["year", "market", "country", "barrier"]],
    eu[["year", "market", "country", "barrier"]],
], ignore_index=True)

agg = (
    combined
    .groupby(["year", "market", "country", "barrier"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

years = list(range(2020, 2026))
markets = ["FDA", "EU"]
countries = ["Vietnam", "Ecuador", "India"]
barriers = ["SPS", "TBT"]

# Create complete skeleton so missing intersections are shown as 0
skeleton = pd.MultiIndex.from_product(
    [years, markets, countries, barriers],
    names=["year", "market", "country", "barrier"],
).to_frame(index=False)

full = skeleton.merge(agg, how="left", on=["year", "market", "country", "barrier"]).fillna({"count": 0})
full["count"] = full["count"].astype(int)

# Pivot to requested wide format (flat columns for export)
pivot = (
    full
    .assign(col=lambda d: d["market"] + "_" + d["country"].map(country_code) + "_" + d["barrier"])
    .pivot(index="year", columns="col", values="count")
    .reindex(years)
)

ordered_cols = []
for market in markets:
    for country in countries:
        for barrier in barriers:
            ordered_cols.append(f"{market}_{country_code[country]}_{barrier}")

pivot = pivot.reindex(columns=ordered_cols).fillna(0).astype(int).reset_index().rename(columns={"year": "Year"})

# -----------------------------
# 4) 2025 partial-data notation
# -----------------------------
max_fda_2025 = fda.loc[fda["year"] == 2025, "date"].max()
max_eu_2025 = eu.loc[eu["year"] == 2025, "date"].max()
latest_2025 = pd.Series([max_fda_2025, max_eu_2025]).max()

pivot_display = pivot.copy()
pivot_display["Year"] = pivot_display["Year"].astype(str)
if pd.notna(latest_2025):
    pivot_display.loc[pivot_display["Year"] == "2025", "Year"] = "2025*"
    footnote = f"2025* - Data as of {latest_2025.strftime('%Y-%m-%d')}"
else:
    footnote = "2025 - No available records in current extracts"

# -----------------------------
# 5) Visualization-friendly table (multi-level columns)
# -----------------------------
visual = pivot_display.set_index("Year").copy()
visual.columns = pd.MultiIndex.from_tuples(
    [tuple(col.split("_", 2)) for col in visual.columns],
    names=["Market", "Country", "Barrier"],
)

# Reorder levels to keep requested hierarchy
visual = visual.reindex(columns=pd.MultiIndex.from_product(
    [["FDA", "EU"], ["VI", "EC", "IN"], ["SPS", "TBT"]
]))

print("Consolidated Statistical Table (easy visualization):")
display(visual)
print("\nNote:")
print(footnote)

# -----------------------------
# 6) Markdown + CSV export
# -----------------------------
def to_markdown_table(df: pd.DataFrame) -> str:
    headers = list(df.columns)
    sep = ["---"] * len(headers)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(sep) + " |",
    ]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(v) for v in row.values) + " |")
    return "\n".join(lines)

markdown_table = to_markdown_table(pivot_display)

export_file = base_dir / "consolidated_shrimp_rejections_2020_2025.csv"
pivot.to_csv(export_file, index=False)

print("\nMarkdown Table:\n")
print(markdown_table)

print("\nCSV Block (copy to Excel):\n")
print(pivot.to_csv(index=False))

print(f"CSV saved to: {export_file}")

Consolidated Statistical Table (easy visualization):


FDA                       EU                    
       VI      EC       IN      VI      EC      IN    
      SPS TBT SPS TBT  SPS TBT SPS TBT SPS TBT SPS TBT
Year                                                  
2020    1   0   0   0  104   0   3   0   0   1   3   0
2021   17   1   0   0   84   0   2   0   3   0   6   0
2022   13   0  34   0   44   1   3   0  13   0   3   0
2023    8   0   8   0   67   1   8   0  34   1   7   2
2024   22   2   2   0   44   3   9   0   8   1   8   2
2025*  23   1   0   0   77   9   6   0  18   0  12   0


Note:
2025* - Data as of 2025-12-17

Markdown Table:

| Year | FDA_VI_SPS | FDA_VI_TBT | FDA_EC_SPS | FDA_EC_TBT | FDA_IN_SPS | FDA_IN_TBT | EU_VI_SPS | EU_VI_TBT | EU_EC_SPS | EU_EC_TBT | EU_IN_SPS | EU_IN_TBT |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2020 | 1 | 0 | 0 | 0 | 104 | 0 | 3 | 0 | 0 | 1 | 3 | 0 |
| 2021 | 17 | 1 | 0 | 0 | 84 | 0 | 2 | 0 | 3 | 0 | 6 | 0 |
| 2022 | 13 | 0 | 34 | 0 | 44 | 1 | 3 | 0 | 13 | 0 | 3 | 0 |
| 2023 | 8 | 0 | 8 | 0 | 67 | 1 | 8 | 0 | 34 | 1 | 7 | 2 |
| 2024 | 22 | 2 | 2 | 0 | 44 | 3 | 9 | 0 | 8 | 1 | 8 | 2 |
| 2025* | 23 | 1 | 0 | 0 | 77 | 9 | 6 | 0 | 18 | 0 | 12 | 0 |

CSV Block (copy to Excel):

Year,FDA_VI_SPS,FDA_VI_TBT,FDA_EC_SPS,FDA_EC_TBT,FDA_IN_SPS,FDA_IN_TBT,EU_VI_SPS,EU_VI_TBT,EU_EC_SPS,EU_EC_TBT,EU_IN_SPS,EU_IN_TBT
2020,1,0,0,0,104,0,3,0,0,1,3,0
2021,17,1,0,0,84,0,2,0,3,0,6,0
2022,13,0,34,0,44,1,3,0,13,0,3,0
2023,8,0,8,0,67,1,8,0,34,1,7,2
2024,22,2,2,0,44,3,9,0,8,1,8,2
2025,23,1,0,0,77,9,6,0,18,0,12